In [4]:
import pandas as pd

In [4]:
dataframe = pd.read_excel("data/创意文案汇总.xlsx")
dataframe.head()

,序号,日期,Query,链接,标注,分值,备注
0,1,5.12,形容武侠三个字,m.baidu,试标,1,NaN
1,2,5.12,广州这座城市的说说,m.baidu,试标,1,NaN
2,3,5.12,吃东西多了的说说心情,m.baidu,试标,1,NaN
3,4,5.12,"老年大学老师身体不佳多时,向他问好怎么说?",m.baidu,试标,1,NaN
4,5,5.12,赞美大门的句子,m.baidu,试标,1,NaN


In [6]:
dataframe = pd.read_excel("data/办公写作_汇总.xlsx")
dataframe.head()

,id,随机Q,链接,返修人,返修,备注,Unnamed: 6
0,5453,村经营性资产情况怎么写,m.baidu,验收,1,NaN,NaN
1,5454,家属放弃搜救的典型事例,m.baidu,验收,0,知识类,NaN
2,5455,三年级二位数乘二位数教学,m.baidu,验收,1,NaN,0.0
3,5456,临床实习经历,m.baidu,验收,1,NaN,NaN
4,5457,向女朋友请假的请假条,m.baidu,验收,1,NaN,NaN


In [8]:
dataframe = pd.read_excel("data/成语标注数据.xlsx")
dataframe.head()

,id,Unnamed: 1,标注,Query,链接,标注.1,备注
0,1,51387随机,试标,静悄悄 abb式,m.baidu,2,NaN
1,2,51387随机,试标,春天的四字成语,m.baidu,2,NaN
2,3,51387随机,试标,什么群什么什么四字成语,m.baidu,2,NaN
3,4,51387随机,试标,又什么又什么的词语,m.baidu,2,NaN
4,5,51387随机,试标,没有抓住重点的成语,m.baidu,1,出现不相关词语：一丝不苟等


In [9]:
dataframe = pd.read_excel("data/诗词标注数据.xlsx")
dataframe.head()

,id,Unnamed: 1,标注,Query,链接,标注.1,备注
0,5001,51378随机,黄显雲,坐于忧患而死于安乐意思,m.baidu,1,NaN
1,5002,51378随机,黄显雲,绝句,m.baidu,2,NaN
2,5003,51378随机,黄显雲,现代诗歌摘抄大全,m.baidu,2,NaN
3,5004,51378随机,黄显雲,王勃的诗,m.baidu,2,NaN
4,5005,51378随机,黄显雲,劝学,m.baidu,2,NaN


In [1]:
#!/usr/bin/env python3
import argparse
import csv
import json
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, Iterable, Iterator, List, Sequence, Tuple
from xml.etree import ElementTree as ET
from zipfile import ZipFile

import jieba
import jieba.posseg as pseg
from paddlenlp import Taskflow
nptag = Taskflow("knowledge_mining", model="nptag", linking=True)


NS_MAIN = "http://schemas.openxmlformats.org/spreadsheetml/2006/main"
NS_REL = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"
NS_PKG_REL = "http://schemas.openxmlformats.org/package/2006/relationships"


def qn(namespace: str, tag: str) -> str:
    """拼接 XML 命名空间和标签名，生成 ElementTree 可识别的完整标签。"""
    return f"{{{namespace}}}{tag}"


def excel_col_to_index(cell_ref: str) -> int:
    """将 Excel 单元格引用中的列标识转换为从 0 开始的列索引。"""
    letters = "".join(ch for ch in cell_ref if ch.isalpha())
    index = 0
    for ch in letters:
        index = index * 26 + ord(ch.upper()) - ord("A") + 1
    return index - 1


def load_shared_strings(workbook: ZipFile) -> List[str]:
    """读取 xlsx 中的 sharedStrings.xml，供字符串单元格解码使用。"""
    shared_strings_path = "xl/sharedStrings.xml"
    if shared_strings_path not in workbook.namelist():
        return []

    root = ET.fromstring(workbook.read(shared_strings_path))
    values = []
    for si in root:
        text = "".join(node.text or "" for node in si.iter(qn(NS_MAIN, "t")))
        values.append(text)
    return values

def resolve_sheet_path(workbook: ZipFile, sheet_name: str) -> str:
    """根据工作表名称解析出对应 worksheet XML 在压缩包中的路径。"""
    # 1. 先打印Excel压缩包里的所有文件，确认真实路径（调试用，可保留）
    print(f"✅ Excel压缩包里的所有文件列表: {workbook.namelist()}")

    # 2. 读取关系文件，获取所有sheet的路径映射，统一去掉前缀冗余
    rel_root = ET.fromstring(workbook.read("xl/_rels/workbook.xml.rels"))
    relationships = {}
    for rel in rel_root.findall(qn(NS_PKG_REL, "Relationship")):
        rel_id = rel.attrib["Id"]
        target_path = rel.attrib["Target"]
        # 核心：去掉路径开头的所有/和xl/，统一为纯相对路径，避免重复拼接
        clean_path = target_path.lstrip("/").lstrip("xl/")
        relationships[rel_id] = clean_path
    print(f"✅ 清洗后的sheet关系映射: {relationships}")

    # 3. 读取workbook.xml，匹配sheet名对应的rel_id
    workbook_root = ET.fromstring(workbook.read("xl/workbook.xml"))
    sheets = workbook_root.find(qn(NS_MAIN, "sheets"))
    if sheets is None:
        raise ValueError("❌ workbook.xml 中未找到 sheets 节点，命名空间可能仍有错误")

    # 先打印所有sheet名，核对是否和你传的--sheet参数一致
    all_sheet_names = [sheet.attrib.get("name", "") for sheet in sheets.findall(qn(NS_MAIN, "sheet"))]
    print(f"✅ Excel文件里的所有工作表名: {all_sheet_names}")

    # 4. 匹配目标sheet，生成正确的唯一路径
    for sheet in sheets.findall(qn(NS_MAIN, "sheet")):
        current_sheet_name = sheet.attrib.get("name", "")
        if current_sheet_name != sheet_name:
            continue
        # 获取sheet对应的关系ID
        rel_id = sheet.attrib.get(qn(NS_REL, "id"))
        if not rel_id or rel_id not in relationships:
            break
        # 核心：只拼接一次xl/，彻底避免双层xl
        sheet_path = f"xl/{relationships[rel_id]}".replace("//", "/")
        # 强校验：路径必须在压缩包的文件列表里
        if sheet_path in workbook.namelist():
            print(f"✅ 匹配到工作表 {sheet_name!r}，正确路径: {sheet_path!r}")
            return sheet_path
        else:
            raise ValueError(f"❌ 工作表 {sheet_name!r} 生成的路径 {sheet_path!r} 不在Excel文件的文件列表中")

    # 没匹配到的情况，抛出明确错误
    raise ValueError(f"❌ 未找到工作表 {sheet_name!r}，Excel里实际的工作表: {all_sheet_names}")


def parse_cell_value(cell: ET.Element, shared_strings: Sequence[str]) -> str:
    """解析单个 Excel 单元格的真实值，兼容共享字符串和内联字符串。"""
    cell_type = cell.attrib.get("t")

    if cell_type == "inlineStr":
        return "".join(node.text or "" for node in cell.iter(qn(NS_MAIN, "t")))

    value_node = cell.find(qn(NS_MAIN, "v"))
    if value_node is None or value_node.text is None:
        return ""

    value = value_node.text
    if cell_type == "s":
        return shared_strings[int(value)]
    return value


def iter_sheet_rows(xlsx_path: Path, sheet_name: str) -> Iterator[Dict[int, str]]:
    """逐行读取指定工作表，并返回按列索引组织的行数据。"""
    with ZipFile(xlsx_path) as workbook:
        shared_strings = load_shared_strings(workbook)
        sheet_path = resolve_sheet_path(workbook, sheet_name)
        sheet_root = ET.fromstring(workbook.read(sheet_path))
        sheet_data = sheet_root.find(qn(NS_MAIN, "sheetData"))
        if sheet_data is None:
            return

        for row in sheet_data.findall(qn(NS_MAIN, "row")):
            values = {}
            for cell in row.findall(qn(NS_MAIN, "c")):
                cell_ref = cell.attrib.get("r", "")
                values[excel_col_to_index(cell_ref)] = parse_cell_value(cell, shared_strings)
            if values:
                yield values


def normalize_label(raw_value: str) -> int:
    """将评分列统一转换为整数标签，兼容 Excel 中的数值格式。"""
    value = str(raw_value).strip()
    if not value:
        raise ValueError("空评分")
    return int(float(value))


def should_keep_token(token: str, keep_punctuation: bool = False) -> bool:
    """判断 jieba 切出的 token 是否应参与统计。"""
    if not token or token.isspace():
        return False
    if keep_punctuation:
        return True
    return not all(unicodedata.category(ch).startswith("P") for ch in token)


def tokenize_min_granularity(
    text: str, keep_punctuation: bool = False, jieba_mode: str = "search"
) -> List[str]:
    """使用 jieba 对 Query 做细粒度切词，并仅返回词结果。"""
    token_pos_pairs = tokenize_with_pos(
        text=text,
        keep_punctuation=keep_punctuation,
        jieba_mode=jieba_mode,
    )
    return [token for token, _ in token_pos_pairs]

def tokenize_with_pos_nptag(original_data):
    """
    模拟 nptag 函数的返回值
    实际使用时替换为真实的 nptag 函数
    """

    # 提取需要处理的词
    words = [item[0] for item in original_data]

    # 调用 nptag 获取结果
    results = nptag(words)

    # 合并原始数据和 nptag 结果
    merged_results = []
    for (token, original_label), nptag_result in zip(original_data, results):
        # 确保 text 字段匹配
        if token == nptag_result['text']:
            new_item = (token, str([original_label, nptag_result['label'], nptag_result['category']]))
            merged_results.append(new_item)

    # 打印结果
    return merged_results

def tokenize_with_pos(
    text: str, keep_punctuation: bool = False, jieba_mode: str = "search"
) -> List[Tuple[str, str]]:
    """使用 jieba 切词并返回词和词性的二元组列表。"""
    normalized = unicodedata.normalize("NFKC", str(text))
    base_token_pos_pairs = []
    for pair in pseg.cut(normalized, HMM=True):
        token = pair.word.strip()
        if should_keep_token(token, keep_punctuation):
            base_token_pos_pairs.append((token, pair.flag))

    if jieba_mode == "accurate":
        return base_token_pos_pairs

    token_pos_pairs = []
    for token, pos in base_token_pos_pairs:
        search_tokens = jieba.lcut_for_search(token, HMM=True)
        for search_token in search_tokens:
            search_token = search_token.strip()
            if should_keep_token(search_token, keep_punctuation):
                token_pos_pairs.append((search_token, pos))
    return token_pos_pairs


def generate_ngrams(tokens: Sequence[str], n: int) -> Iterable[Tuple[str, ...]]:
    """基于切词结果生成连续 ngram，供短语粒度统计使用。"""
    if len(tokens) < n:
        return []
    return (tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1))


def generate_ngram_with_pos(
    token_pos_pairs: Sequence[Tuple[str, str]], n: int
) -> Iterable[Tuple[Tuple[str, ...], Tuple[str, ...]]]:
    """基于带词性的切词结果生成 ngram 及对应的词性序列。"""
    if len(token_pos_pairs) < n:
        return []
    return (
        (
            tuple(token for token, _ in token_pos_pairs[i : i + n]),
            tuple(pos for _, pos in token_pos_pairs[i : i + n]),
        )
        for i in range(len(token_pos_pairs) - n + 1)
    )


def ratio(numerator: int, denominator: int) -> float:
    """安全计算占比，避免分母为 0 时抛错。"""
    if denominator == 0:
        return 0.0
    return numerator / denominator


def update_item_stats(
    items: Sequence,
    label: int,
    occurrence_stats: Dict,
    query_stats: Dict,
) -> None:
    """更新 token 或 ngram 的出现次数统计和 Query 去重统计。"""
    occurrence_counter = Counter(items)
    for item, count in occurrence_counter.items():
        occurrence_stats[item][label] += count
        query_stats[item][label] += 1


def update_pos_stats(item_pos_pairs: Sequence[Tuple[object, object]], pos_stats: Dict) -> None:
    """统计每个 token 或 ngram 对应的词性分布。"""
    occurrence_counter = Counter(item_pos_pairs)
    for (item, pos), count in occurrence_counter.items():
        pos_stats[item][pos] += count


def get_primary_pos(pos_counter: Counter) -> str:
    """返回某个 token 或 ngram 出现次数最多的词性标注。"""
    if not pos_counter:
        return ""
    top_pos, _ = max(pos_counter.items(), key=lambda item: (item[1], str(item[0])))
    if isinstance(top_pos, tuple):
        return "/".join(top_pos)
    return str(top_pos)


def build_output_rows(
    stats_occ: Dict,
    stats_query: Dict,
    pos_stats: Dict,
    item_type: str,
    include_label_2: bool = True,
) -> List[Dict[str, object]]:
    """将内部统计结果整理为可直接写入 CSV 的表格结构。"""
    rows = []
    for item, query_counts in stats_query.items():
        query_0 = query_counts.get(0, 0)
        query_1 = query_counts.get(1, 0)
        query_2 = query_counts.get(2, 0)
        query_01 = query_0 + query_1
        if query_01 == 0:
            continue

        occ_counts = stats_occ[item]
        occ_0 = occ_counts.get(0, 0)
        occ_1 = occ_counts.get(1, 0)
        occ_2 = occ_counts.get(2, 0)
        occ_01 = occ_0 + occ_1

        token_list = list(item) if isinstance(item, tuple) else [item]
        pos = get_primary_pos(pos_stats.get(item, Counter()))
        row = {
            item_type: "".join(token_list),
            "tokens": "/".join(token_list),
            "pos": pos,
            "n": len(token_list),
            "query_count_0": query_0,
            "query_count_1": query_1,
            "query_count_0_1": query_01,
            "query_ratio_0_in_0_1": round(ratio(query_0, query_01), 6),
            "query_ratio_1_in_0_1": round(ratio(query_1, query_01), 6),
            "occurrence_count_0": occ_0,
            "occurrence_count_1": occ_1,
            "occurrence_count_0_1": occ_01,
            "occurrence_ratio_0_in_0_1": round(ratio(occ_0, occ_01), 6),
            "occurrence_ratio_1_in_0_1": round(ratio(occ_1, occ_01), 6),
        }
        if include_label_2:
            row["query_count_2"] = query_2
            row["occurrence_count_2"] = occ_2
        rows.append(row)

    rows.sort(
        key=lambda row: (
            row["n"],
            -row["query_count_0_1"],
            -row["occurrence_count_0_1"],
            row[item_type],
        )
    )
    return rows


def write_csv(path: Path, rows: List[Dict[str, object]], fieldnames: Sequence[str]) -> None:
    """将统计结果按指定字段顺序写入 CSV 文件。"""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8-sig", newline="") as output_file:
        writer = csv.DictWriter(output_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def analyze(
    xlsx_path: Path,
    sheet_name: str,
    query_column: str,
    label_column: str,
    output_dir: Path,
    min_ngram: int,
    max_ngram: int,
    keep_punctuation: bool,
    jieba_mode: str,
) -> Dict[str, object]:
    """执行完整分析流程，输出 token/ngram 统计表和汇总信息。"""
    rows_iter = iter_sheet_rows(xlsx_path, sheet_name)
    try:
        header_row = next(rows_iter)
    except StopIteration as exc:
        raise ValueError("Excel 中没有可读取的数据。") from exc

    header_map = {str(value).strip(): index for index, value in header_row.items() if str(value).strip()}
    if query_column not in header_map:
        raise ValueError(f"未找到 Query 列 {query_column!r}，实际表头: {list(header_map)}")
    if label_column not in header_map:
        raise ValueError(f"未找到评分列 {label_column!r}，实际表头: {list(header_map)}")

    query_index = header_map[query_column]
    label_index = header_map[label_column]

    label_distribution = Counter()
    token_occurrence_stats = defaultdict(Counter)
    token_query_stats = defaultdict(Counter)
    token_pos_stats = defaultdict(Counter)
    ngram_occurrence_stats = defaultdict(Counter)
    ngram_query_stats = defaultdict(Counter)
    ngram_pos_stats = defaultdict(Counter)

    total_rows = 0
    valid_rows = 0

    for row in rows_iter:
        # 只处理同时具备 Query 和评分的数据行。
        total_rows += 1
        query = str(row.get(query_index, "")).strip()
        label_raw = row.get(label_index, "")
        if not query or str(label_raw).strip() == "":
            continue

        try:
            label = normalize_label(label_raw)
        except ValueError:
            continue

        if label not in {0, 1, 2}:
            continue

        valid_rows += 1
        label_distribution[label] += 1

        # 先做 jieba 切词和词性标注，再基于切词结果统计单词和短语级分布。
        token_par_results = tokenize_with_pos(
            query,
            keep_punctuation=keep_punctuation,
            jieba_mode=jieba_mode,
        )
        try:
            token_pos_pairs = tokenize_with_pos_nptag(token_par_results)
        except:
            continue
            
        print("tokenize_with_pos", query, keep_punctuation, jieba_mode, token_pos_pairs)

        tokens = [token for token, _ in token_pos_pairs]

        update_item_stats(tokens, label, token_occurrence_stats, token_query_stats)
        update_pos_stats(token_pos_pairs, token_pos_stats)
        for n in range(min_ngram, max_ngram + 1):
            ngrams = list(generate_ngrams(tokens, n))
            ngram_pos_pairs = list(generate_ngram_with_pos(token_pos_pairs, n))
            if ngrams:
                update_item_stats(ngrams, label, ngram_occurrence_stats, ngram_query_stats)
            if ngram_pos_pairs:
                update_pos_stats(ngram_pos_pairs, ngram_pos_stats)

    token_rows = build_output_rows(
        token_occurrence_stats,
        token_query_stats,
        token_pos_stats,
        item_type="token",
    )
    ngram_rows = build_output_rows(
        ngram_occurrence_stats,
        ngram_query_stats,
        ngram_pos_stats,
        item_type="ngram",
    )

    token_fields = [
        "token",
        "tokens",
        "pos",
        "n",
        "query_count_0",
        "query_count_1",
        "query_count_2",
        "query_count_0_1",
        "query_ratio_0_in_0_1",
        "query_ratio_1_in_0_1",
        "occurrence_count_0",
        "occurrence_count_1",
        "occurrence_count_2",
        "occurrence_count_0_1",
        "occurrence_ratio_0_in_0_1",
        "occurrence_ratio_1_in_0_1",
    ]
    ngram_fields = [
        "ngram",
        "tokens",
        "pos",
        "n",
        "query_count_0",
        "query_count_1",
        "query_count_2",
        "query_count_0_1",
        "query_ratio_0_in_0_1",
        "query_ratio_1_in_0_1",
        "occurrence_count_0",
        "occurrence_count_1",
        "occurrence_count_2",
        "occurrence_count_0_1",
        "occurrence_ratio_0_in_0_1",
        "occurrence_ratio_1_in_0_1",
    ]

    write_csv(output_dir / "token_stats.csv", token_rows, token_fields)
    write_csv(output_dir / "ngram_stats.csv", ngram_rows, ngram_fields)

    summary = {
        "xlsx_path": str(xlsx_path),
        "sheet_name": sheet_name,
        "query_column": query_column,
        "label_column": label_column,
        "total_rows": total_rows,
        "valid_rows": valid_rows,
        "label_distribution": dict(sorted(label_distribution.items())),
        "token_stats_rows": len(token_rows),
        "ngram_stats_rows": len(ngram_rows),
        "token_output": str(output_dir / "token_stats.csv"),
        "ngram_output": str(output_dir / "ngram_stats.csv"),
        "tokenization": f"jieba切词；模式={jieba_mode}；默认过滤纯标点 token。",
        "pos_tagging": "词性来自 jieba.posseg；search 模式下扩展子词继承原始切词词性。",
        "ngram_range": [min_ngram, max_ngram],
    }
    with (output_dir / "summary.json").open("w", encoding="utf-8") as output_file:
        json.dump(summary, output_file, ensure_ascii=False, indent=2)
    return summary




/root/paddlejob/workspace/env_run/output/zacharychu/miniconda3/envs/paddle/lib/python3.12/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/root/paddlejob/workspace/env_run/output/zacharychu/miniconda3/envs/paddle/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

def parse_args() -> argparse.Namespace:
    """解析命令行参数，支持切词模式和 ngram 范围配置。"""
    parser = argparse.ArgumentParser(
        description="读取 Query 和评分列，按 jieba 细粒度切词，并统计 token/ngram 在标签 0 和 1 下的占比。"
    )
    # env_run/output/zacharychu/code/star_static/data/工作写作标注结果.xlsx

    # -------------------------------------------
    parser.add_argument(
        "--input",
        default="data/创意文案汇总.xlsx",
        help="输入 Excel 文件路径。",
    )
    parser.add_argument("-f", default="pass", help="pass")
    parser.add_argument("--sheet", default="Sheet1", help="工作表名称，默认使用“汇总”。")
    parser.add_argument("--query-column", default="Query", help="Query 列名。")
    parser.add_argument("--label-column", default="分值", help="评分列名。")
    parser.add_argument(
        "--output-dir",
        default="output/shortText_ngram_stats",
    )
    # --------------------------------------------------
    parser.add_argument("--min-ngram", type=int, default=2, help="ngram 最小长度，默认 2。")
    parser.add_argument("--max-ngram", type=int, default=4, help="ngram 最大长度，默认 4。")
    parser.add_argument(
        "--jieba-mode",
        choices=["search", "accurate"],
        default="accurate",
        help="jieba 切词模式。search 更细，accurate 更稳，默认 search。",
    )
    parser.add_argument(
        "--keep-punctuation",
        action="store_true",
        help="保留纯标点 token。默认过滤纯标点。",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    print(f"启动参数: {args}")
    if args.min_ngram < 2:
        raise ValueError("--min-ngram 不能小于 2，单词统计已包含在 token 结果中。")
    if args.max_ngram < args.min_ngram:
        raise ValueError("--max-ngram 不能小于 --min-ngram。")

    for excelPath in list_file("data/资源号随机query集合/ori_data"):
        excel_path = Path(excelPath)
        if not excel_path.exists():
            print(f"⚠️ 文件不存在: {excel_path}")
            continue
        # 安全生成输出目录，避免文件名切割错误
        file_name = excel_path.stem
        output_dir = Path("output") / f"{file_name}_shortText_ngram_stats"
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f"📂 开始处理文件: {excel_path}，输出目录: {output_dir}")

        try:
            summary = analyze(
                xlsx_path=excel_path,
                sheet_name=args.sheet,
                query_column=args.query_column,
                label_column=args.label_column,
                output_dir=output_dir,
                min_ngram=args.min_ngram,
                max_ngram=args.max_ngram,
                keep_punctuation=args.keep_punctuation,
                jieba_mode=args.jieba_mode,
            )
            print(f"✅ 文件处理完成: {excel_path}")
            print(json.dumps(summary, ensure_ascii=False, indent=2))
        except Exception as e:
            print(f"❌ 文件处理失败: {excel_path}，错误信息: {str(e)}")
            continue


if __name__ == "__main__":
    main()

tokenize_with_pos 迪迦群名霸气简短 False accurate [('迪迦群', "['nr', '数量词', '生活用语类']"), ('名', "['q', '数量词', '生活用语类']"), ('霸气', "['n', '个体描述', '生活用语类']"), ('简短', "['v', '作品', '作品类_实体']")]
tokenize_with_pos 劝红颜知己工作太累的暖心句子 False accurate [('劝', "['v', '单字', '生活用语类']"), ('红颜', "['nr', '作品', '作品类_实体']"), ('知己', "['n', '称呼', '人物类_概念']"), ('工作', "['vn', '作品', '作品类_实体']"), ('太', "['d', '单字', '生活用语类']"), ('累', "['a', '单字', '生活用语类']"), ('的', "['uj', '单字', '生活用语类']"), ('暖心', "['n', '情绪', '生活用语类']"), ('句子', "['n', '教育用语', '术语类_教育用语']")]
tokenize_with_pos 栩栩栩如生造句 False accurate [('栩', "['i', '单字', '生活用语类']"), ('栩栩如生', "['i', '作品', '作品类_实体']"), ('造句', "['n', '教育用语', '术语类_教育用语']")]
tokenize_with_pos 生活就是在忙碌中寻找乐趣的句子 False accurate [('生活', "['vn', '作品', '作品类_实体']"), ('就是', "['d', '作品', '作品类_实体']"), ('在', "['p', '单字', '生活用语类']"), ('忙碌', "['a', '生活用语', '生活用语类']"), ('中', "['f', '单字', '生活用语类']"), ('寻找', "['v', '作品', '作品类_实体']"), ('乐趣', "['a', '生活用语', '生活用语类']"), ('的', "['uj', '单字', '生活用语类']"), ('句子', "['n', '教育用语', 

In [ ]:
def parse_args() -> argparse.Namespace:
    """解析命令行参数，支持切词模式和 ngram 范围配置。"""
    parser = argparse.ArgumentParser(
        description="读取 Query 和评分列，按 jieba 细粒度切词，并统计 token/ngram 在标签 0 和 1 下的占比。"
    )
    # env_run/output/zacharychu/code/star_static/data/工作写作标注结果.xlsx

    # -------------------------------------------
    parser.add_argument(
        "--input",
        default="data/办公写作_汇总.xlsx",
        help="输入 Excel 文件路径。",
    )
    parser.add_argument("-f", default="pass", help="pass")
    parser.add_argument("--sheet", default="Sheet1", help="工作表名称，默认使用“汇总”。")
    parser.add_argument("--query-column", default="随机Q", help="Query 列名。")
    parser.add_argument("--label-column", default="返修", help="评分列名。")
    parser.add_argument(
        "--output-dir",
        default="output/workText_ngram_stats",
    )
    # --------------------------------------------------
    parser.add_argument("--min-ngram", type=int, default=2, help="ngram 最小长度，默认 2。")
    parser.add_argument("--max-ngram", type=int, default=4, help="ngram 最大长度，默认 4。")
    parser.add_argument(
        "--jieba-mode",
        choices=["search", "accurate"],
        default="accurate",
        help="jieba 切词模式。search 更细，accurate 更稳，默认 search。",
    )
    parser.add_argument(
        "--keep-punctuation",
        action="store_true",
        help="保留纯标点 token。默认过滤纯标点。",
    )
    return parser.parse_args()


def main() -> None:
    """脚本入口，校验参数后执行分析并打印汇总结果。"""
    
    args = parse_args()
    print(args)
    if args.min_ngram < 2:
        raise ValueError("--min-ngram 不能小于 2，单词统计已包含在 token 结果中。")
    
    if args.max_ngram < args.min_ngram:
        raise ValueError("--max-ngram 不能小于 --min-ngram。")

    summary = analyze(
        xlsx_path=Path(args.input),
        sheet_name=args.sheet,
        query_column=args.query_column,
        label_column=args.label_column,
        output_dir=Path(args.output_dir),
        min_ngram=args.min_ngram,
        max_ngram=args.max_ngram,
        keep_punctuation=args.keep_punctuation,
        jieba_mode=args.jieba_mode,
    )

    print(json.dumps(summary, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()

tokenize_with_pos 小学生家访记录表内容怎么写 False accurate [('小学生', "['nr', '称呼', '人物类_概念']"), ('家访', "['j', '生活用语', '生活用语类']"), ('记录表', "['n', '文件', '文化类_制度政策协议']"), ('内容', "['n', '生活用语', '生活用语类']"), ('怎么', "['r', '作品', '作品类_实体']"), ('写', "['v', '单字', '生活用语类']")]
tokenize_with_pos 自流坪施工方案 False accurate [('自流', "['n', '修饰词', '生活用语类']"), ('坪', "['ns', '单字', '生活用语类']"), ('施工', "['vn', '生活用语', '生活用语类']"), ('方案', "['n', '方案', '生活用语类']")]
tokenize_with_pos 小学生午睡条模板 False accurate [('小学生', "['nr', '称呼', '人物类_概念']"), ('午睡', "['v', '生活用语', '生活用语类']"), ('条', "['n', '数量词', '生活用语类']"), ('模板', "['n', '材料', '物体类']")]
tokenize_with_pos 真理诞生于一百个问号之后的教案 False accurate [('真理', "['n', '生活用语', '生活用语类']"), ('诞生', "['v', '作品', '作品类_实体']"), ('于', "['p', '单字', '生活用语类']"), ('一百个', "['m', '数量词', '生活用语类']"), ('问号', "['n', '符号', '术语类_符号指标类']"), ('之后', "['f', '时间', '时间类']"), ('的', "['uj', '单字', '生活用语类']"), ('教案', "['n', '生活用语', '生活用语类']")]
tokenize_with_pos 旅游加好友通过申请怎么写 False accurate [('旅游', "['vn', '旅游', '生活用语类']"), ('加',

In [31]:
tokenize_with_pos_nptag(results)

[('静悄悄', "['z', '作品', '作品类_实体']"), ('成语', "['n', '作品类型', '作品类_概念']")]

In [14]:
def parse_args() -> argparse.Namespace:
    """解析命令行参数，支持切词模式和 ngram 范围配置。"""
    parser = argparse.ArgumentParser(
        description="读取 Query 和评分列，按 jieba 细粒度切词，并统计 token/ngram 在标签 0 和 1 下的占比。"
    )
    # env_run/output/zacharychu/code/star_static/data/工作写作标注结果.xlsx

    # -------------------------------------------
    parser.add_argument(
        "--input",
        default="data/成语标注数据.xlsx",
        help="输入 Excel 文件路径。",
    )
    parser.add_argument("-f", default="pass", help="pass")
    parser.add_argument("--sheet", default="Sheet1", help="工作表名称，默认使用“汇总”。")
    parser.add_argument("--query-column", default="Query", help="Query 列名。")
    parser.add_argument("--label-column", default="label", help="评分列名。")
    parser.add_argument(
        "--output-dir",
        default="output/hanyu_51387chengyu_ngram_stats",
    )
    # --------------------------------------------------
    parser.add_argument("--min-ngram", type=int, default=2, help="ngram 最小长度，默认 2。")
    parser.add_argument("--max-ngram", type=int, default=4, help="ngram 最大长度，默认 4。")
    parser.add_argument(
        "--jieba-mode",
        choices=["search", "accurate"],
        default="accurate",
        help="jieba 切词模式。search 更细，accurate 更稳，默认 search。",
    )
    parser.add_argument(
        "--keep-punctuation",
        action="store_true",
        help="保留纯标点 token。默认过滤纯标点。",
    )
    return parser.parse_args()


def main() -> None:
    """脚本入口，校验参数后执行分析并打印汇总结果。"""
    
    args = parse_args()
    print(args)
    if args.min_ngram < 2:
        raise ValueError("--min-ngram 不能小于 2，单词统计已包含在 token 结果中。")
    
    if args.max_ngram < args.min_ngram:
        raise ValueError("--max-ngram 不能小于 --min-ngram。")

    summary = analyze(
        xlsx_path=Path(args.input),
        sheet_name=args.sheet,
        query_column=args.query_column,
        label_column=args.label_column,
        output_dir=Path(args.output_dir),
        min_ngram=args.min_ngram,
        max_ngram=args.max_ngram,
        keep_punctuation=args.keep_punctuation,
        jieba_mode=args.jieba_mode,
    )

    print(json.dumps(summary, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()

Building prefix dict from the default dictionary ...
[2026-05-19 02:09:22,126] [   DEBUG] __init__.py:113 - Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
[2026-05-19 02:09:22,128] [   DEBUG] __init__.py:132 - Loading model from cache /tmp/jieba.cache


Namespace(input='data/汉语_51387成语标注数据.xlsx', f='/root/.local/share/jupyter/runtime/kernel-42243fbe-b216-4e0c-b464-ecbe8f5c13a6.json', sheet='Sheet1', query_column='Query', label_column='label', output_dir='output/hanyu_51387chengyu_ngram_stats', min_ngram=2, max_ngram=4, jieba_mode='accurate', keep_punctuation=False)


Loading model cost 0.719 seconds.
[2026-05-19 02:09:22,847] [   DEBUG] __init__.py:164 - Loading model cost 0.719 seconds.
Prefix dict has been built successfully.
[2026-05-19 02:09:22,848] [   DEBUG] __init__.py:166 - Prefix dict has been built successfully.
/root/paddlejob/workspace/env_run/output/zacharychu/miniconda3/envs/paddle/lib/python3.12/site-packages/paddlenlp/transformers/tokenizer_utils_base.py:1899: UserWarning: Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
  warnings.warn(


tokenize_with_pos 成语相什么, False accurate [('成语', "['n', '作品类型', '作品类_概念']"), ('相', "['v', '单字', '生活用语类']"), ('什么', "['r', '作品', '作品类_实体']")]
tokenize_with_pos 四字成语关心 False accurate [('四', "['m', '数量词', '生活用语类']"), ('字', "['n', '单字', '生活用语类']"), ('成语', "['n', '作品类型', '作品类_概念']"), ('关心', "['n', '态度', '生活用语类']")]
tokenize_with_pos 形容长途跋涉的艰辛的词语 False accurate [('形容', "['n', '生活用语', '生活用语类']"), ('长途跋涉', "['i', '作品', '作品类_实体']"), ('的', "['uj', '单字', '生活用语类']"), ('艰辛', "['a', '生活用语', '生活用语类']"), ('的', "['uj', '单字', '生活用语类']"), ('词语', "['n', '术语', '术语类']")]
tokenize_with_pos 用周写一个成语 False accurate [('用', "['p', '单字', '生活用语类']"), ('周', "['nr', '数量词', '生活用语类']"), ('写', "['v', '单字', '生活用语类']"), ('一个', "['m', '数量词', '生活用语类']"), ('成语', "['n', '作品类型', '作品类_概念']")]
tokenize_with_pos 穷的成语 False accurate [('穷', "['a', '单字', '生活用语类']"), ('的', "['uj', '单字', '生活用语类']"), ('成语', "['n', '作品类型', '作品类_概念']")]
tokenize_with_pos 救济还能组什么词 False accurate [('救济', "['n', '术语', '术语类']"), ('还', "['d', '单字', '生活用语类']"),

In [15]:
dataframe.columns

Index(['id', 'Unnamed: 1', '标注', 'Query', '链接', '标注.1', '备注'], dtype='str')

In [18]:
def parse_args() -> argparse.Namespace:
    """解析命令行参数，支持切词模式和 ngram 范围配置。"""
    parser = argparse.ArgumentParser(
        description="读取 Query 和评分列，按 jieba 细粒度切词，并统计 token/ngram 在标签 0 和 1 下的占比。"
    )
    # env_run/output/zacharychu/code/star_static/data/工作写作标注结果.xlsx

    # -------------------------------------------
    parser.add_argument(
        "--input",
        default="data/诗词标注数据.xlsx",
        help="输入 Excel 文件路径。",
    )
    parser.add_argument("-f", default="pass", help="pass")
    parser.add_argument("--sheet", default="汇总", help="工作表名称，默认使用“汇总”。")
    parser.add_argument("--query-column", default="Query", help="Query 列名。")
    parser.add_argument("--label-column", default="lable", help="评分列名。")
    parser.add_argument(
        "--output-dir",
        default="output/shici_ngram_stats",
    )
    # --------------------------------------------------
    parser.add_argument("--min-ngram", type=int, default=2, help="ngram 最小长度，默认 2。")
    parser.add_argument("--max-ngram", type=int, default=4, help="ngram 最大长度，默认 4。")
    parser.add_argument(
        "--jieba-mode",
        choices=["search", "accurate"],
        default="accurate",
        help="jieba 切词模式。search 更细，accurate 更稳，默认 search。",
    )
    parser.add_argument(
        "--keep-punctuation",
        action="store_true",
        help="保留纯标点 token。默认过滤纯标点。",
    )
    return parser.parse_args()


def main() -> None:
    """脚本入口，校验参数后执行分析并打印汇总结果。"""
    
    args = parse_args()
    print(args)
    if args.min_ngram < 2:
        raise ValueError("--min-ngram 不能小于 2，单词统计已包含在 token 结果中。")
    
    if args.max_ngram < args.min_ngram:
        raise ValueError("--max-ngram 不能小于 --min-ngram。")

    summary = analyze(
        xlsx_path=Path(args.input),
        sheet_name=args.sheet,
        query_column=args.query_column,
        label_column=args.label_column,
        output_dir=Path(args.output_dir),
        min_ngram=args.min_ngram,
        max_ngram=args.max_ngram,
        keep_punctuation=args.keep_punctuation,
        jieba_mode=args.jieba_mode,
    )

    print(json.dumps(summary, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()

tokenize_with_pos 描写江南美景的诗词 False accurate [('描写', "['v', '生活用语', '生活用语类']"), ('江南', "['ns', '地区', '世界地区类']"), ('美景', "['n', '现象', '生活用语类']"), ('的', "['uj', '单字', '生活用语类']"), ('诗词', "['n', '诗歌类型', '作品类_概念']")]
tokenize_with_pos 怎么简短的现代诗 False accurate [('怎么', "['r', '作品', '作品类_实体']"), ('简短', "['v', '作品', '作品类_实体']"), ('的', "['uj', '单字', '生活用语类']"), ('现代诗', "['n', '诗歌类型', '作品类_概念']")]
tokenize_with_pos 行舟的古诗 False accurate [('行舟', "['n', '作品', '作品类_实体']"), ('的', "['uj', '单字', '生活用语类']"), ('古诗', "['n', '诗歌类型', '作品类_概念']")]
tokenize_with_pos 行到水穷处坐看云起时啥意思 False accurate [('行', "['v', '单字', '生活用语类']"), ('到', "['v', '单字', '生活用语类']"), ('水', "['n', '单字', '生活用语类']"), ('穷', "['a', '单字', '生活用语类']"), ('处', "['n', '单字', '生活用语类']"), ('坐', "['v', '单字', '生活用语类']"), ('看', "['v', '单字', '生活用语类']"), ('云', "['n', '单字', '生活用语类']"), ('起', "['v', '单字', '生活用语类']"), ('时', "['ng', '单字', '生活用语类']"), ('啥意思', "['n', '作品', '作品类_实体']")]
tokenize_with_pos 谭嗣侗个人诗词 False accurate [('谭嗣侗', "['nr', '人', '人物类_实体']"), ('个人